# Plugin

> LavaSR v2 speech enhancement plugin — provides bandwidth extension and optional denoising to improve speech audio quality before transcription.

In [ ]:
#| default_exp plugin

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
import logging
import uuid
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch

from cjm_media_plugin_system.processing_interface import MediaProcessingPlugin
from cjm_media_plugin_system.core import MediaMetadata
from cjm_media_plugin_system.storage import MediaProcessingStorage

from cjm_plugin_system.utils.hashing import hash_file
from cjm_plugin_system.core.interface import RELOAD_TRIGGER, plugin_action, collect_plugin_actions
from cjm_plugin_system.core.errors import (
    PluginInputError, PluginResourceError, ResourceShortfall,
)
from cjm_plugin_system.utils.validation import (
    dict_to_config, config_to_dict, dataclass_to_jsonschema,
    SCHEMA_TITLE, SCHEMA_DESC, SCHEMA_ENUM,
)

## Configuration

In [ ]:
#| export
@dataclass
class LavaSRPluginConfig:
    """Configuration for the LavaSR speech enhancement plugin."""
    
    model_path: str = field(
        default="YatharthS/LavaSR",
        metadata={
            SCHEMA_TITLE: "Model Path",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "HuggingFace model ID or local path for LavaSR weights."
        }
    )
    
    device: str = field(
        default="auto",
        metadata={
            SCHEMA_TITLE: "Device",
            RELOAD_TRIGGER: "model",  # CR-4: change triggers model reload
            SCHEMA_DESC: "Compute device. 'auto' selects CUDA if available.",
            SCHEMA_ENUM: ["auto", "cpu", "cuda"]
        }
    )
    
    denoise: bool = field(
        default=True,
        metadata={
            SCHEMA_TITLE: "Denoise",
            SCHEMA_DESC: "Apply UL-UNAS denoising before enhancement. Recommended for noisy audio."
        }
    )
    
    enhance: bool = field(
        default=True,
        metadata={
            SCHEMA_TITLE: "Enhance (BWE)",
            SCHEMA_DESC: "Apply bandwidth extension enhancement. Upsamples and improves clarity."
        }
    )
    
    batch_mode: bool = field(
        default=True,
        metadata={
            SCHEMA_TITLE: "Batch Mode",
            SCHEMA_DESC: "Process audio in 1.28-second batches. Recommended for segments longer than ~30 seconds to manage memory."
        }
    )
    
    input_sr: int = field(
        default=16000,
        metadata={
            SCHEMA_TITLE: "Input Sample Rate",
            SCHEMA_DESC: "Sample rate for LavaSR's internal representation (Hz). 16000 is the default and recommended value."
        }
    )
    
    cutoff: Optional[int] = field(
        default=None,
        metadata={
            SCHEMA_TITLE: "Cutoff Frequency",
            SCHEMA_DESC: "Frequency cutoff for Linkwitz-Riley filter (Hz). None = auto (input_sr / 2)."
        }
    )
    
    output_format: str = field(
        default="wav",
        metadata={
            SCHEMA_TITLE: "Output Format",
            SCHEMA_DESC: "Format for output audio files. WAV recommended for lossless 48kHz output.",
            SCHEMA_ENUM: ["wav", "flac", "mp3"]
        }
    )

## Plugin Class

In [ ]:
#| export
class LavaSRProcessingPlugin(MediaProcessingPlugin):
    """LavaSR v2 speech enhancement plugin for bandwidth extension and denoising."""
    
    config_class = LavaSRPluginConfig
    
    OUTPUT_SAMPLE_RATE = 48000  # LavaSR always outputs at 48kHz
    
    def __init__(self):
        """Initialize the plugin."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: Optional[LavaSRPluginConfig] = None
        self.storage: Optional[MediaProcessingStorage] = None
        self._data_dir: Optional[str] = None
        self._model = None  # Lazy-loaded LavaEnhance2 instance
    
    # ── Properties ──────────────────────────────────────────────────
    
    @property
    def name(self) -> str:  # Plugin name identifier
        """Get the plugin name."""
        return "cjm-media-plugin-lavasr"
    
    @property
    def version(self) -> str:  # Plugin version string
        """Get the plugin version."""
        from cjm_media_plugin_lavasr import __version__
        return __version__
    
    @property
    def supported_media_types(self) -> List[str]:  # Supported media types
        """Get supported media types."""
        return ["audio"]
    
    # ── Lifecycle ────────────────────────────────────────────────────
    
    def _apply_config(self,
                      config: Optional[Any] = None,  # Configuration dict or None for defaults
                     ) -> None:
        """CR-4: apply config values only. Called by initialize (first-time) and the
        substrate's reconfigure delta path. Model release on a model_path/device change
        is handled declaratively via RELOAD_TRIGGER -> _release_model (device resolved
        lazily in _load_model)."""
        self.config = dict_to_config(LavaSRPluginConfig, config or {})

    def initialize(self,
                   config: Optional[Any] = None,  # Configuration dict or None for defaults
                  ) -> None:
        """First-time setup. CR-4: config application factored into _apply_config; the
        substrate's reconfigure path fires _release_model on a model_path/device change
        then re-applies config."""
        self._apply_config(config)
        
        from .meta import get_plugin_metadata
        metadata = get_plugin_metadata()
        db_path = metadata["db_path"]
        self._data_dir = str(Path(db_path).parent)
        
        self.storage = MediaProcessingStorage(db_path)
        self.logger.info(f"Initialized with device={self.config.device}, "
                         f"denoise={self.config.denoise}, enhance={self.config.enhance}")
    
    def prefetch(self) -> None:
        """CR-4 (SG-19): eagerly load the model so the first execute() doesn't pay
        the download/load cost. Idempotent via _load_model's None-guard."""
        self._load_model()

    def on_disable(self) -> None:
        """CR-2: release the GPU model when the operator disables the plugin (the
        worker stays alive); lazy reload on the next execute after re-enable."""
        self._release_model()

    def cleanup(self) -> None:
        """Clean up plugin resources."""
        self._release_model()
        self.logger.info("Plugin cleaned up")
    
    def is_available(self) -> bool:  # Whether the plugin can run
        """Check if the plugin is available on this system."""
        try:
            import LavaSR  # noqa
            return True
        except ImportError:
            return False
    
    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for UI forms
        """Return JSON Schema for the plugin configuration."""
        return dataclass_to_jsonschema(LavaSRPluginConfig)
    
    def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
        """Return the current configuration."""
        return config_to_dict(self.config) if self.config else {}
    
    # ── Model Management ────────────────────────────────────────────
    
    def _load_model(self) -> None:
        """Load the LavaSR v2 model (lazy, cached)."""
        if self._model is not None:
            return
        
        device = self.config.device
        if device == "auto":
            device = "cuda" if torch.cuda.is_available() else "cpu"
        
        self.logger.info(f"Loading LavaSR v2 model from '{self.config.model_path}' on {device}...")
        from LavaSR.model import LavaEnhance2
        self._model = LavaEnhance2(self.config.model_path, device=device)
        self.logger.info("LavaSR v2 model loaded")
    
    def _release_model(self) -> None:
        """Unload the LavaSR model and free GPU memory."""
        if self._model is not None:
            del self._model
            self._model = None
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            self.logger.info("Model unloaded, CUDA cache cleared")
    
    # ── Job Storage ──────────────────────────────────────────────────
    
    def _store_job(self,
                   action: str,  # Action name
                   input_path: str,  # Source file path
                   output_path: str,  # Output file path
                   parameters: Optional[Dict[str, Any]] = None,  # Action parameters
                   metadata: Optional[Dict[str, Any]] = None,  # Additional metadata
                   input_hash: Optional[str] = None,  # Pre-computed input hash
                  ) -> str:  # Generated job_id
        """Hash input/output files and store a processing job record."""
        job_id = str(uuid.uuid4())
        if input_hash is None:
            input_hash = hash_file(input_path)
        output_hash = hash_file(output_path)
        self.storage.save(
            job_id=job_id, action=action,
            input_path=input_path, input_hash=input_hash,
            output_path=output_path, output_hash=output_hash,
            parameters=parameters, metadata=metadata
        )
        return job_id
    
    # ── Action Dispatch ──────────────────────────────────────────────
    
    def execute(self,
                action: str = "enhance_speech",  # Action to perform
                **kwargs
               ) -> Dict[str, Any]:  # Action result
        """Dispatch to the `@plugin_action`-tagged handler for `action` (SG-44)."""
        for klass in type(self).__mro__:
            for attr in vars(klass).values():
                if getattr(attr, "_plugin_action", None) == action:
                    return attr(self, **kwargs)
        raise PluginInputError(  # SG-47: typed input-validation
            f"Unknown action: {action}", fields_invalid=["action"],
        )

    @plugin_action("get_info")
    def _action_get_info(self, **kwargs) -> Dict[str, Any]:
        """Action wrapper -> get_info()."""
        return self.get_info(kwargs["file_path"]).to_dict()

    @plugin_action("enhance_speech")
    def _action_enhance_speech(self, **kwargs) -> Dict[str, Any]:
        """Action wrapper -> _enhance_speech()."""
        return self._enhance_speech(**kwargs)
    
    # ── Interface Methods (not applicable) ───────────────────────────
    
    def get_info(self,
                 file_path: str,  # Path to audio file
                ) -> MediaMetadata:  # File metadata
        """Get basic audio file metadata via soundfile."""
        import soundfile as sf
        info = sf.info(file_path)
        return MediaMetadata(
            path=file_path,
            format=Path(file_path).suffix.lstrip('.'),
            duration=info.duration,
            size_bytes=Path(file_path).stat().st_size,
            audio_streams=[{
                'channels': info.channels,
                'sample_rate': info.samplerate,
            }],
            video_streams=[],
        )
    
    def convert(self, input_path, output_format, **kwargs):
        """Not applicable for speech enhancement."""
        raise PluginInputError(  # SG-47: typed input-validation; this method is
            # not applicable for this plugin domain.
            "convert is not supported by the LavaSR plugin. "
            "Use 'enhance_speech' instead.",
            fields_invalid=["action"],
        )
    
    def extract_segment(self, input_path, start, end, output_path=None):
        """Not applicable for speech enhancement."""
        raise PluginInputError(  # SG-47: typed input-validation; this method is
            # not applicable for this plugin domain.
            "extract_segment is not supported by the LavaSR plugin. "
            "Use 'enhance_speech' instead.",
            fields_invalid=["action"],
        )
    
    # ── Core Action ──────────────────────────────────────────────────
    
    def _enhance_speech(self,
                        input_path: str,  # Path to audio segment file
                        output_dir: Optional[str] = None,  # Output directory (default: data_dir/enhanced/)
                        output_format: Optional[str] = None,  # Output format override
                        denoise: Optional[bool] = None,  # Override config's denoise setting
                       ) -> Dict[str, Any]:  # Enhancement result
        """Enhance speech quality via bandwidth extension and optional denoising."""
        import soundfile as sf
        
        fmt = output_format or self.config.output_format
        use_denoise = denoise if denoise is not None else self.config.denoise
        input_p = Path(input_path)
        
        # Determine output directory
        if output_dir is None:
            out_dir = Path(self._data_dir) / "enhanced" / input_p.stem
        else:
            out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        
        # Load model (lazy, cached)
        self.report_progress(0.0, "Loading model...")
        self._load_model()
        
        # Load audio via LavaSR (resamples to input_sr → 16kHz, sets cutoff)
        self.report_progress(0.1, "Loading audio...")
        input_audio, input_sr = self._model.load_audio(
            str(input_p),
            input_sr=self.config.input_sr,
            cutoff=self.config.cutoff,
        )
        
        # Enhance. SG-47 Track B wraps the inference site so CUDA OOM surfaces
        # as PluginResourceError → CR-7 reactive-retry reloads.
        self.report_progress(0.2, "Enhancing speech...")
        try:
            output_audio = self._model.enhance(
                input_audio,
                enhance=self.config.enhance,
                denoise=use_denoise,
                batch=self.config.batch_mode,
            )
        except torch.cuda.OutOfMemoryError as e:
            free_bytes = torch.cuda.mem_get_info()[0] if torch.cuda.is_available() else 0
            available_mb = free_bytes / (1024 ** 2)
            raise PluginResourceError(
                f"CUDA OOM during LavaSR enhance: {e}",
                resource_shortfall=ResourceShortfall(
                    resource='gpu_vram_mb',
                    needed=available_mb + 100.0,
                    available=available_mb,
                ),
            ) from e
        
        # Save output (always 48kHz)
        self.report_progress(0.8, "Saving output...")
        output_path = out_dir / f"enhanced.{fmt}"
        output_np = output_audio.cpu().numpy().squeeze()
        sf.write(str(output_path), output_np, self.OUTPUT_SAMPLE_RATE)
        
        # Get duration from output
        duration = len(output_np) / self.OUTPUT_SAMPLE_RATE
        
        # Resolve device for metadata
        device = self.config.device
        if device == "auto":
            device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # Hash and store job
        self.report_progress(0.9, "Storing job record...")
        job_id = self._store_job(
            action="enhance_speech",
            input_path=str(input_p),
            output_path=str(output_path),
            parameters={
                "denoise": use_denoise,
                "enhance": self.config.enhance,
                "batch_mode": self.config.batch_mode,
                "input_sr": self.config.input_sr,
                "cutoff": self.config.cutoff,
                "output_format": fmt,
            },
            metadata={
                "input_sr": self.config.input_sr,
                "output_sr": self.OUTPUT_SAMPLE_RATE,
                "duration": duration,
                "device": device,
            },
        )
        
        self.report_progress(1.0, "Complete")
        
        return {
            "job_id": job_id,
            "output_path": str(output_path),
            "input_path": str(input_p),
            "input_sample_rate": self.config.input_sr,
            "output_sample_rate": self.OUTPUT_SAMPLE_RATE,
            "duration": duration,
            "denoise_applied": use_denoise,
            "enhance_applied": self.config.enhance,
        }


LavaSRProcessingPlugin.supported_actions = collect_plugin_actions(LavaSRProcessingPlugin)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()